# RSANME: Numerical Experiments Example

This notebook demonstrates how to use the RSANME (Relaxed Self-Adaptive Newton Method with Error) algorithm for solving nonlinear equations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rsanme import RSANME, modified_newton_method
from test_problems import Problem1, Problem3

%matplotlib inline

## Example 1: Simple System

Let's solve a simple 2D nonlinear system:
- F1(x) = x1^2 + x2^2 - 1
- F2(x) = x1 - x2

In [ ]:
def func_simple(x):
    return np.array([x[0]**2 + x[1]**2 - 1, x[0] - x[1]])

def jacobian_simple(x):
    return np.array([[2*x[0], 2*x[1]], [1, -1]])

x0 = np.array([0.5, 0.5])

# Solve with RSANME
solver = RSANME(func_simple, jacobian_simple, x0, alpha=0.7)
x_sol, info = solver.solve()

print(f"Solution: {x_sol}")
print(f"Converged: {info['converged']}")
print(f"Iterations: {info['num_iterations']}")
print(f"Final residual: {info['residuals'][-1]:.2e}")

## Example 2: Convergence Visualization

Let's visualize the convergence behavior on the Broyden System.

In [ ]:
problem = Problem1()

# Test different alpha values
alphas = [0.3, 0.5, 0.7]
plt.figure(figsize=(10, 6))

for alpha in alphas:
    solver = RSANME(
        func=problem.func,
        jacobian=problem.jacobian,
        x0=problem.initial_guess(),
        alpha=alpha
    )
    _, info = solver.solve()
    plt.semilogy(info['residuals'], marker='o', label=f'RSANME (α={alpha})')

# Compare with Newton
_, info_newton = modified_newton_method(
    func=problem.func,
    jacobian=problem.jacobian,
    x0=problem.initial_guess()
)
plt.semilogy(info_newton['residuals'], marker='s', label='Newton', linewidth=2)

plt.xlabel('Iteration')
plt.ylabel('Residual Norm ||F(x)||')
plt.title('Convergence Comparison: Broyden System')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Example 3: Parameter Study

Let's study how different relaxation parameters affect convergence.

In [ ]:
problem = Problem3(5)
alphas = np.linspace(0.1, 0.9, 9)
iterations = []
residuals = []

for alpha in alphas:
    solver = RSANME(
        func=problem.func,
        jacobian=problem.jacobian,
        x0=problem.initial_guess(),
        alpha=alpha
    )
    x_sol, info = solver.solve()
    
    if info['converged']:
        iterations.append(info['num_iterations'])
        residuals.append(info['residuals'][-1])
    else:
        iterations.append(100)
        residuals.append(info['residuals'][-1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(alphas, iterations, marker='o', linewidth=2)
ax1.set_xlabel('Relaxation Parameter α')
ax1.set_ylabel('Iterations to Converge')
ax1.set_title('Effect of Relaxation Parameter on Iterations')
ax1.grid(True, alpha=0.3)

ax2.semilogy(alphas, residuals, marker='o', linewidth=2)
ax2.set_xlabel('Relaxation Parameter α')
ax2.set_ylabel('Final Residual')
ax2.set_title('Effect of Relaxation Parameter on Residual')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Example 4: Running Full Experiments

Run comprehensive experiments and display results.

In [ ]:
from test_problems import get_test_problems
import pandas as pd

# Run experiments on all problems
problems = get_test_problems()[:5]  # First 5 problems
results = []

for problem in problems:
    print(f"Testing {problem.name}...")
    
    # Test RSANME
    solver = RSANME(
        func=problem.func,
        jacobian=problem.jacobian,
        x0=problem.initial_guess(),
        alpha=0.7
    )
    x_sol, info = solver.solve()
    
    results.append({
        'Problem': problem.name,
        'Method': 'RSANME',
        'Converged': '✓' if info['converged'] else '✗',
        'Iterations': info['num_iterations'],
        'Final Residual': f"{info['residuals'][-1]:.2e}"
    })
    
    # Test Newton
    x_sol, info = modified_newton_method(
        func=problem.func,
        jacobian=problem.jacobian,
        x0=problem.initial_guess()
    )
    
    results.append({
        'Problem': problem.name,
        'Method': 'Newton',
        'Converged': '✓' if info['converged'] else '✗',
        'Iterations': info['num_iterations'],
        'Final Residual': f"{info['residuals'][-1]:.2e}"
    })

df = pd.DataFrame(results)
print("\nResults Summary:")
print(df.to_string(index=False))

## Conclusions

The RSANME algorithm demonstrates:
1. Robust convergence on various problem types
2. Adjustable convergence speed via relaxation parameter
3. Self-adaptive step sizes for improved stability
4. Competitive performance compared to classical Newton method